# Modeling & Evaluation — Previsão do Vencedor de Confrontos Profissionais de CS2

**Projeto:** Previsão do vencedor de confrontos profissionais de CS2
**Dataset:** [Counter-Strike Pro Matches (Kaggle, ektarr)](https://www.kaggle.com/datasets/ektarr/counter-strike-pro-matches)
**Base:** Machine Learning Canvas + notebooks de Data Understanding e Data Preparation

Este notebook treina e compara **vários modelos de classificação diferentes** para prever se a equipa 1 vence (`target = 1`) ou perde (`target = 0`) cada jogo/mapa.

### Pontos de partida (vindos do notebook de Data Preparation)
- Ficheiros de entrada: `cs2_matches_train.csv` e `cs2_matches_test.csv`, já com split **cronológico** (80/20) e sem *leakage*.
- 12 features pré-jogo: **Elo** (`elo_team1`, `elo_team2`, `elo_diff`), taxa histórica de vitórias, forma recente (últimos 5 jogos) e confrontos diretos (h2h).
- O **Elo é usado como feature de treino**, tal como as restantes. Não há baseline Elo e, nas métricas, aparece apenas o desempenho dos modelos.

### Metodologia
1. Carregar e validar os dados.
2. Definir os modelos e as respetivas grelhas de hiperparâmetros.
3. Afinar cada modelo com `GridSearchCV` e **validação cruzada temporal** (`TimeSeriesSplit`), para nunca treinar com o futuro.
4. Avaliar todos no conjunto de teste (jogos mais recentes).
5. Comparar, escolher o melhor modelo, analisar erros e importância das features.
6. Guardar resultados e o melhor modelo.

## 1. Setup

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
RANDOM_STATE = 42

## 2. Carregamento dos dados

Lê os dois ficheiros gerados pelo notebook de Data Preparation. No Colab, se não estiverem na pasta atual, é pedido o upload.

In [ ]:
FICHEIROS = ['cs2_matches_train.csv', 'cs2_matches_test.csv']

if not all(os.path.exists(f) for f in FICHEIROS):
    try:
        from google.colab import files
        print("Carrega os ficheiros cs2_matches_train.csv e cs2_matches_test.csv (gerados no Data Preparation).")
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Faltam ficheiros: {[f for f in FICHEIROS if not os.path.exists(f)]}")

train = pd.read_csv('cs2_matches_train.csv')
test = pd.read_csv('cs2_matches_test.csv')

X_train, y_train = train.drop(columns=['target']), train['target']
X_test, y_test = test.drop(columns=['target']), test['target']

assert list(X_train.columns) == list(X_test.columns), "Treino e teste com colunas diferentes"
assert not X_train.isna().any().any() and not X_test.isna().any().any(), "Existem valores em falta"

print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")
print(f"\nFeatures ({X_train.shape[1]}): {list(X_train.columns)}")
print("\nProporção do alvo (treino / teste):")
print(pd.concat([y_train.value_counts(normalize=True).rename('treino'),
                 y_test.value_counts(normalize=True).rename('teste')], axis=1).round(3))

## 3. Modelos e grelhas de hiperparâmetros

Famílias de modelos diferentes, para comparar abordagens distintas:

| Família | Modelos |
|---|---|
| Linear | Regressão Logística |
| Baseada em distância | KNN |
| Margem máxima | SVM (kernel RBF) |
| Árvore única | Decision Tree |
| *Bagging* de árvores | Random Forest, Extra Trees |
| *Boosting* de árvores | Gradient Boosting, HistGradientBoosting, XGBoost*, LightGBM* |
| Rede neuronal | MLP |

\* XGBoost e LightGBM entram automaticamente se estiverem instalados (no Colab já vêm instalados).

Os modelos sensíveis à escala (Regressão Logística, KNN, SVM, MLP) vão dentro de um `Pipeline` com `StandardScaler`, que é ajustado só com os dados de treino de cada fold.

In [ ]:
def com_escala(modelo):
    return Pipeline([('scaler', StandardScaler()), ('clf', modelo)])

# nome -> (estimador, grelha de hiperparâmetros)
modelos = {
    'Regressão Logística': (
        com_escala(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        {'clf__C': [0.01, 0.1, 1, 10]}),

    'KNN': (
        com_escala(KNeighborsClassifier()),
        {'clf__n_neighbors': [15, 31, 61], 'clf__weights': ['uniform', 'distance']}),

    'SVM (RBF)': (
        com_escala(SVC(kernel='rbf', random_state=RANDOM_STATE)),
        {'clf__C': [0.1, 1], 'clf__gamma': ['scale', 0.1]}),

    'Decision Tree': (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {'max_depth': [3, 5, 8], 'min_samples_leaf': [20, 50]}),

    'Random Forest': (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {'n_estimators': [200, 400], 'max_depth': [5, 10, None], 'min_samples_leaf': [5, 20]}),

    'Extra Trees': (
        ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {'n_estimators': [200, 400], 'max_depth': [5, 10, None], 'min_samples_leaf': [5, 20]}),

    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [2, 3]}),

    'HistGradientBoosting': (
        HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        {'learning_rate': [0.03, 0.1], 'max_depth': [3, 6], 'min_samples_leaf': [20, 50]}),

    'MLP (rede neuronal)': (
        com_escala(MLPClassifier(max_iter=500, early_stopping=True, random_state=RANDOM_STATE)),
        {'clf__hidden_layer_sizes': [(32,), (64, 32)], 'clf__alpha': [1e-3, 1e-2]}),
}

try:
    from xgboost import XGBClassifier
    modelos['XGBoost'] = (
        XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1),
        {'n_estimators': [100, 300], 'learning_rate': [0.03, 0.1], 'max_depth': [2, 4]})
except ImportError:
    print("XGBoost não está instalado — ignorado.")

try:
    from lightgbm import LGBMClassifier
    modelos['LightGBM'] = (
        LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
        {'n_estimators': [100, 300], 'learning_rate': [0.03, 0.1], 'num_leaves': [7, 15]})
except ImportError:
    print("LightGBM não está instalado — ignorado.")

print(f"{len(modelos)} modelos definidos:", list(modelos))

## 4. Treino com validação cruzada temporal

`TimeSeriesSplit` usa janelas expansivas: cada fold treina nos jogos mais antigos e valida nos seguintes. Assim, a seleção de hiperparâmetros respeita a ordem cronológica, tal como o split treino/teste. A métrica de seleção é a **accuracy**.

In [ ]:
cv_temporal = TimeSeriesSplit(n_splits=5)

melhores = {}      # nome -> estimador já ajustado com os melhores hiperparâmetros
registo_cv = []

for nome, (estimador, grelha) in modelos.items():
    t0 = time.time()
    gs = GridSearchCV(estimador, grelha, cv=cv_temporal, scoring='accuracy', n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    melhores[nome] = gs.best_estimator_
    registo_cv.append({
        'Modelo': nome,
        'Accuracy CV (média)': gs.best_score_,
        'Melhores hiperparâmetros': gs.best_params_,
        'Tempo (s)': round(time.time() - t0, 1),
    })
    print(f"{nome:<22} CV accuracy = {gs.best_score_:.4f}  ({time.time() - t0:.1f}s)")

df_cv = pd.DataFrame(registo_cv).sort_values('Accuracy CV (média)', ascending=False).reset_index(drop=True)
df_cv

## 5. Avaliação no conjunto de teste

Cada modelo (já ajustado com a totalidade do treino e os melhores hiperparâmetros) prevê os jogos mais recentes, que nunca viu. Métricas: accuracy, precision, recall, F1 e ROC AUC.

In [ ]:
def pontuacao(modelo, X):
    # Probabilidade da classe 1; para modelos sem predict_proba usa-se a função de decisão
    if hasattr(modelo, 'predict_proba'):
        return modelo.predict_proba(X)[:, 1]
    return modelo.decision_function(X)

linhas = []
previsoes = {}
for nome, modelo in melhores.items():
    y_pred = modelo.predict(X_test)
    previsoes[nome] = y_pred
    linhas.append({
        'Modelo': nome,
        'Accuracy (treino)': accuracy_score(y_train, modelo.predict(X_train)),
        'Accuracy (teste)': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC AUC': roc_auc_score(y_test, pontuacao(modelo, X_test)),
    })

resultados = (pd.DataFrame(linhas)
              .sort_values('Accuracy (teste)', ascending=False)
              .reset_index(drop=True))
resultados['Gap treino-teste'] = resultados['Accuracy (treino)'] - resultados['Accuracy (teste)']
resultados.round(4)

### 5.1 Comparação visual

In [ ]:
ordem = resultados.sort_values('Accuracy (teste)')
fig, ax = plt.subplots(figsize=(9, 5))
barras = ax.barh(ordem['Modelo'], ordem['Accuracy (teste)'], color='#4C78A8')
ax.bar_label(barras, fmt='%.3f', padding=3)
ax.set_xlim(max(0, ordem['Accuracy (teste)'].min() - 0.05), min(1, ordem['Accuracy (teste)'].max() + 0.05))
ax.set_xlabel('Accuracy no conjunto de teste')
ax.set_title('Comparação de modelos — accuracy de teste')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(resultados))
ax.bar(x - 0.2, resultados['Accuracy (treino)'], width=0.4, label='Treino', color='#9ecae9')
ax.bar(x + 0.2, resultados['Accuracy (teste)'], width=0.4, label='Teste', color='#4C78A8')
ax.set_xticks(x)
ax.set_xticklabels(resultados['Modelo'], rotation=35, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Treino vs teste (diferença grande = overfitting)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Melhor modelo

Matriz de confusão e importância das features (por permutação, medida no conjunto de teste).

In [ ]:
nome_melhor = resultados.loc[0, 'Modelo']
melhor = melhores[nome_melhor]
print(f"Melhor modelo no teste: {nome_melhor}")
print(f"Accuracy de teste: {resultados.loc[0, 'Accuracy (teste)']:.4f}")
print(f"Hiperparâmetros: {df_cv.set_index('Modelo').loc[nome_melhor, 'Melhores hiperparâmetros']}")

fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, previsoes[nome_melhor]),
                       display_labels=['Equipa 2 vence', 'Equipa 1 vence']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Matriz de confusão — {nome_melhor}')
plt.tight_layout()
plt.show()

In [ ]:
perm = permutation_importance(melhor, X_test, y_test, scoring='accuracy',
                              n_repeats=15, random_state=RANDOM_STATE, n_jobs=-1)
imp = (pd.DataFrame({'Feature': X_test.columns,
                     'Queda de accuracy': perm.importances_mean,
                     'Desvio-padrão': perm.importances_std})
       .sort_values('Queda de accuracy', ascending=True))

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(imp['Feature'], imp['Queda de accuracy'], xerr=imp['Desvio-padrão'], color='#F58518')
ax.set_xlabel('Queda média de accuracy ao baralhar a feature')
ax.set_title(f'Importância das features (permutação) — {nome_melhor}')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

imp.sort_values('Queda de accuracy', ascending=False).reset_index(drop=True).round(4)

## 7. Guardar resultados e melhor modelo

In [ ]:
resultados.round(4).to_csv('resultados_modelos_cs2.csv', index=False)
df_cv.to_csv('resultados_cv_cs2.csv', index=False)
joblib.dump(melhor, 'melhor_modelo_cs2.joblib')

print("Ficheiros guardados: resultados_modelos_cs2.csv, resultados_cv_cs2.csv, melhor_modelo_cs2.joblib")
print(f"Melhor modelo: {nome_melhor}")

## 8. Notas para a discussão no relatório

- **Teto de desempenho:** prever vencedores no CS2 profissional é difícil. Mesmo o melhor modelo deve ficar apenas moderadamente acima do acaso; diferenças de 1–2 pontos de accuracy entre modelos podem estar dentro do ruído (o teste tem ~2 mil jogos, logo a margem de erro ronda os ±2 pontos percentuais).
- **Overfitting:** a coluna `Gap treino-teste` mostra que modelos flexíveis (Random Forest, Extra Trees, boosting) tendem a decorar o treino; o gap ajuda a justificar a escolha do modelo final.
- **Validação:** o CV temporal e o split cronológico garantem que nenhum modelo usa informação futura.
- **Granularidade:** os modelos preveem o vencedor de cada **mapa**. Para prever o confronto completo (BO3/BO5) é preciso agregar as previsões por `match_id` (por exemplo, a equipa com mais mapas previstos vence) — passo possível numa fase seguinte, mas exige manter `match_id` nos ficheiros preparados.
- **Possíveis melhorias:** adicionar mapa e formato (`bestOf`) às features — no notebook de Data Preparation, `colunas_map` e `colunas_bestof` saíram vazias, por isso não entraram —, afinar o `k` do Elo, usar torneio como feature e testar um Elo específico por mapa.